# Text generation and constraints

The synthetic corpus is small, so disable overlap filtering for demonstration only. For real corpora use test_output=True and inspect novelty.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import markovify
from markovify.nlp import Corpus
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUT = ROOT / "notebooks" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
print(markovify.__file__)

In [ ]:
model = markovify.load_model(OUT / "compiled.json.gz", random_state=SEED)
opts = dict(test_output=False, tries=100, max_words=30)
examples = {
    "free":model.make_sentence(**opts),
    "prefix":model.make_sentence(start_with="La proteína", **opts),
    "suffix":model.make_sentence(end_with="durante el ensayo.", **opts),
    "all":model.make_sentence(include=["proteína","receptor"], **opts),
    "any":model.make_sentence(include=["proteína","ligando"], match="any", **opts),
    "phrase":model.make_sentence_around("con el receptor", **opts),
    "exclude":model.make_sentence(exclude=["enzima"], **opts),
}
display(pd.Series(examples))

In [ ]:
generated = model.make_sentences(100, unique=True, max_attempts=500, **opts)
train = Corpus.load(OUT / "train.jsonl")
known = set(train.texts)
print("Generated:",len(generated))
print("Exact training overlap:",sum(s in known for s in generated)/len(generated) if generated else float("nan"))
Corpus(generated).save(OUT / "generated.jsonl")
if generated:
    pd.Series([len(s.split()) for s in generated]).plot.hist(bins=15,title="Generated sentence lengths")
    plt.xlabel("Tokens")
    plt.show()
display(pd.DataFrame({"sentence":generated[:20]}))